In [ ]:
#%% md
# Текстовый энкодер новостей MOEX — финальная версия
#
# ЦЕЛЬ: этот ноутбук — текстовая половина системы "новости -> движение цены". На выходе —
# таблица (событие x тикер) с эмбеддингом, весом значимости, весом новизны и классом
# "времени действия" новости. Дальше эта таблица склеивается с ценовым рядом во ВТОРОМ
# ноутбуке (модель котировок) — сюда рыночные данные сознательно НЕ включены.
#
# Структура:
#   Часть 1 (без изменений по логике, уже отлажена) — дедупликация: точные совпадения,
#       TF-IDF, ModernBERT + адаптивный порог, FAISS-сверка. Несколько методов сравниваются
#       намеренно (учебная ценность), из них выбирается SELECTED_METHOD.
#   Часть 2 (новое) — обогащение новостей метаданными для энкодера:
#     - Значимость (significance) — вес по кластеру дублей / числу уникальных источников
#       (ваше предложение, с уточнением — см. раздел 2.2).
#     - Новизна (novelty / Event Similarity Gap) — дополняет значимость: перепечатка в
#       первый час важнее, чем то же самое в 10-й раз через 3 недели.
#     - Длительность эффекта (duration/decay) — новая часть по вашему запросу: смена
#       руководства компании -> недели, разовая цитата политика -> часы.
#     - Релевантность тикера (REL) — нужна, чтобы понимать, ПО КАКОМУ тикеру считать вес,
#       если в одной новости упомянуто несколько компаний.
#
# Что осознанно НЕ входит сюда (и почему):
#   - Регрессия на волатильность/объём IMOEX — задача СЛЕДУЮЩЕГО ноутбука (модель котировок).
#   - Сравнение эмбеддинг-моделей (RuModernBERT vs BGE-M3), потоковая обработка,
#     LLM-валидация кластеров — остаются отдельными исследовательскими ноутбуками (02-04
#     из прошлого пакета); здесь используется уже выбранная связка RuModernBERT-small +
#     адаптивный порог, как в проверенном базовом пайплайне.
#%%


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#%% md
# Фильтрация и дедупликация новостей

#Сравниваются четыре гипотезы: точный дубль, дубль после нормализации, почти одинаковый текст (TF-IDF) и семантический дубль с **ModernBERT**. Кластеры формируются только из публикаций, вышедших не далее чем через 14 дней друг от друга.
#%%


from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('deduplication_results')
OUT_DIR.mkdir(exist_ok=True)
MODEL_NAME = 'deepvk/RuModernBERT-small'
WINDOW_DAYS = 14
pd.set_option('display.max_colwidth', 180)

In [ ]:
import torch
if torch.cuda.is_available():
    print(True)
    print(torch.cuda.get_device_name(0))  # Покажет модель GPU
else:
    print(False)
    print('CUDA недоступна — вычисления пойдут на CPU')

In [ ]:
#%% md
## 1. Загрузка данных
#Читаем CSV и проверяем объём и временной диапазон данных.
#%%
df = pd.read_csv(DATA_PATH, parse_dates=['post_dttm']).reset_index(drop=True)
print(f'Новостей: {len(df):,}; период: {df.post_dttm.min():%d.%m.%Y}—{df.post_dttm.max():%d.%m.%Y}')
df[['id', 'source', 'post_dttm', 'data']].head(3)

In [ ]:
## 2. Нормализация текста
#Удаляем URL, упоминания каналов, различия регистра и лишнюю пунктуацию.
#%%
def normalize_text(text):
    """Удаляет технические различия, не удаляя факты, числа и тикеры."""
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df['text_normalized'] = df['data'].fillna('').map(normalize_text)
df['text_length'] = df['text_normalized'].str.len()
# Пустые тексты не считаем дублями друг друга.
df = df[df.text_length.gt(0)].copy().reset_index(drop=True)
# Стабильный числовой id строки — понадобится в Части 2 для привязки эмбеддингов.
df['_row_id'] = np.arange(len(df))
len(df)

In [ ]:
## 3. Формирование кластеров
#Объединяем связанные пары похожих новостей в кластеры и считаем метрики экспериментов.
#%%
class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.parent[b] = a

def clusters_from_edges(n, edges):
    uf = UnionFind(n)
    for a, b in edges: uf.union(a, b)
    return pd.factorize([uf.find(i) for i in range(n)])[0]

def exact_clusters(series):
    return pd.factorize(series)[0]

def top_k_edges(vectors, dates, threshold, n_neighbors=6, batch_size=128):
    """бачевый top-K поиск без матрицы похожести N×N в памяти (19220x19220)много расчетов """
    ts = pd.to_datetime(dates, utc=True).astype('int64').to_numpy()
    max_delta = pd.Timedelta(days=WINDOW_DAYS).value
    edges = []
    for start in tqdm(range(0, len(ts), batch_size), desc=f'Поиск пар ≥ {threshold:.2f}', leave=False):
        stop = min(start + batch_size, len(ts))
        sim = vectors[start:stop] @ vectors.T
        if hasattr(sim, 'toarray'): sim = sim.toarray()
        idx = np.argpartition(sim, -n_neighbors, axis=1)[:, -n_neighbors:]
        for local_i, candidates in enumerate(idx):
            i = start + local_i
            for j in candidates:
                if i < j and sim[local_i, j] >= threshold and abs(ts[i] - ts[j]) <= max_delta:
                    edges.append((i, int(j)))
    return edges

def method_stats(name, clusters):
    sizes = pd.Series(clusters).value_counts()
    removed = int((sizes - 1).clip(lower=0).sum())
    return dict(method=name, duplicate_groups=int((sizes > 1).sum()),
                duplicate_posts=int(sizes[sizes > 1].sum()), posts_to_remove=removed,
                remaining_posts=len(df) - removed, largest_group=int(sizes.max()))

In [ ]:
#%% md
## 4. Базовые гипотезы дедупликации
#Сравниваем точные совпадения, нормализованные совпадения и почти идентичные тексты через TF-IDF.
#%%

experiments = {}
experiments['exact_raw'] = exact_clusters(df.data.fillna(''))
experiments['exact_normalized'] = exact_clusters(df.text_normalized)

# символьный TF-IDF устойчив к небольшим правкам текста. почитай за analyzer
tfidf = TfidfVectorizer(analyzer='word', ngram_range=(2, 5), min_df=4, max_features=500_000)
print('Строю TF-IDF-признаки…')
tfidf_vectors = tfidf.fit_transform(tqdm(df.text_normalized, desc='TF-IDF: подготовка текстов'))
for threshold in tqdm((0.90, 0.95), desc='TF-IDF: пороги'):
    name = f'tfidf_{threshold:.2f}'
    experiments[name] = clusters_from_edges(
        len(df), top_k_edges(tfidf_vectors, df.post_dttm, threshold)
    )

report = pd.DataFrame([method_stats(name, clusters) for name, clusters in experiments.items()])
report.sort_values('posts_to_remove', ascending=False)

In [ ]:
#%% md
## 5. Проверка найденных дублей
#Выбираем метод и просматриваем кандидатов на исключение до сохранения результата.
#%%
SELECTED_METHOD = 'tfidf_0.90'  # или exact_raw / exact_normalized / tfidf_0.95

# Добавляем номера кластеров к строкам датафрейма
df_view = df.assign(duplicate_cluster=experiments[SELECTED_METHOD])

# В каждом кластере сохраняем самый ранний пост,
# последующие — кандидаты на удаление
ordered = df_view.sort_values(['duplicate_cluster', 'post_dttm'])
posts_to_remove = ordered[
    ordered.duplicated('duplicate_cluster', keep='first')
].copy()

print(f'Метод: {SELECTED_METHOD}')
print(f'Постов к исключению: {len(posts_to_remove):,}')

display(
    posts_to_remove[
        ['duplicate_cluster', 'id', 'source', 'post_dttm', 'data']
    ].head(30)
)

In [ ]:
#%% md
## Семантическая гипотеза — ModernBERT


#%%
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# Загружаем только токенайзер — модель и GPU для этой проверки не нужны
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Считаем число токенов без truncation, батчами
token_lengths = []

for start in tqdm(
    range(0, len(df), 256),
    desc='Подсчёт токенов'
):
    texts_batch = df['text_normalized'].iloc[start:start + 256].tolist()

    encoded = tokenizer(
        texts_batch,
        add_special_tokens=True,
        truncation=False,
        padding=False
    )
    
    token_lengths.extend(len(ids) for ids in encoded['input_ids'])

df['token_length'] = token_lengths

quantiles = df['token_length'].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
).round().astype(int)

display(quantiles.rename('число токенов').to_frame())

print(
    f"Постов, обрезаемых при max_length=512:  {(df['token_length'] > 512).mean():.2%}\n"
    f"Постов, обрезаемых при max_length=1024: {(df['token_length'] > 1024).mean():.2%}\n"
    f"Постов, обрезаемых при max_length=2048: {(df['token_length'] > 2048).mean():.2%}"
)
#%%


In [ ]:
p99 = df['token_length'].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(
    df.loc[df['token_length'] <= p99, 'token_length'],
    bins=60,
    ax=axes[0]
)
axes[0].axvline(512, color='orange', linestyle='--', label='512')
axes[0].axvline(1024, color='red', linestyle='--', label='1024')
axes[0].set(
    title='Распределение длины новостей в токенах (до 99-го перцентиля)',
    xlabel='Токенов',
    ylabel='Количество новостей'
)
axes[0].legend()

sorted_lengths = np.sort(df['token_length'])
cdf = np.arange(1, len(sorted_lengths) + 1) / len(sorted_lengths)

axes[1].plot(sorted_lengths, cdf, linewidth=2)
for length in (256, 512, 1024, 2048):
    share = (df['token_length'] <= length).mean()
    axes[1].axvline(length, color='gray', linestyle='--', alpha=0.6)
    axes[1].annotate(
        f'{length}: {share:.1%}',
        xy=(length, share),
        xytext=(length, min(share + 0.06, 0.98)),
        ha='center'
    )

axes[1].set(
    title='Накопленная доля покрытых новостей',
    xlabel='max_length',
    ylabel='Доля новостей без обрезания',
    ylim=(0, 1.05)
)
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()
#%%


In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

def modernbert_embeddings(texts, batch_size=128):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Устройство: {device}; модель: {MODEL_NAME}')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()
    result = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), batch_size), desc='ModernBERT: подготовка текстов'):
            batch = tokenizer(texts[start:start + batch_size], padding=True, truncation=True,
                              max_length=1024, return_tensors='pt').to(device)
            hidden = model(**batch).last_hidden_state
            mask = batch['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)
            result.append(torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy())
    return np.vstack(result)

emb_path = OUT_DIR / 'modernbert_embeddings.npy'
if emb_path.exists():
    embeddings = np.load(emb_path)
else:
    embeddings = modernbert_embeddings(df.text_normalized.tolist())
    np.save(emb_path, embeddings)
embeddings.shape

In [ ]:

# Чем выше порог, тем меньше риск склеить разные новости
# ============================================================================
# АДАПТИВНЫЙ ПОИСК ОПТИМАЛЬНОГО ПОРОГА ДЕДУПЛИКАЦИИ
# ============================================================================
# Этот код заменяет ручной перебор порогов (0.87, 0.90, 0.93)
# Он автоматически находит лучший порог на основе метрики удалённых новостей.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from scipy.signal import savgol_filter

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.parent[b] = a

def clusters_from_edges(n, edges):
    uf = UnionFind(n)
    for a, b in edges: uf.union(a, b)
    return pd.factorize([uf.find(i) for i in range(n)])[0]

def top_k_edges(vectors, dates, threshold, n_neighbors=6, batch_size=128):
    ts = pd.to_datetime(dates, utc=True).astype('int64').to_numpy()
    max_delta = pd.Timedelta(days=WINDOW_DAYS).value
    edges = []
    for start in tqdm(range(0, len(ts), batch_size), desc=f'Поиск пар ≥ {threshold:.2f}', leave=False):
        stop = min(start + batch_size, len(ts))
        sim = vectors[start:stop] @ vectors.T
        if hasattr(sim, 'toarray'): sim = sim.toarray()
        idx = np.argpartition(sim, -n_neighbors, axis=1)[:, -n_neighbors:]
        for local_i, candidates in enumerate(idx):
            i = start + local_i
            for j in candidates:
                if i < j and sim[local_i, j] >= threshold and abs(ts[i] - ts[j]) <= max_delta:
                    edges.append((i, int(j)))
    return edges

def compute_duplicate_stats(threshold, embeddings, dates, top_k=6):
    edges = top_k_edges(embeddings, dates, threshold, n_neighbors=top_k)
    clusters = clusters_from_edges(len(df), edges)
    sizes = pd.Series(clusters).value_counts()
    dup_sizes = sizes[sizes > 1]
    num_dup_clusters = len(dup_sizes)
    total_duplicates = dup_sizes.sum()
    removed = (dup_sizes - 1).sum()
    avg_dup_size = dup_sizes.mean() if len(dup_sizes) > 0 else 0
    if len(dup_sizes) > 1:
        dup_mask = np.isin(clusters, dup_sizes.index)
        if dup_mask.sum() > 1:
            X_clean = embeddings[dup_mask]
            labels_clean = clusters[dup_mask]
            try:
                sil = silhouette_score(X_clean, labels_clean, metric='cosine')
            except ValueError:
                sil = -1
        else:
            sil = -1
    else:
        sil = -1
    return {'threshold': threshold, 'num_dup_clusters': num_dup_clusters,
            'total_duplicates': total_duplicates, 'removed': removed,
            'avg_dup_size': avg_dup_size, 'silhouette': sil, 'clusters': clusters}

def adaptive_threshold_search(embeddings, dates,
                              initial_step=0.1,
                              min_step=0.001,
                              change_threshold=0.05,
                              max_iter=50,
                              top_k=6):
    thresholds = [0.0, 0.5, 1.0]
    results = {}
    for t in thresholds:
        results[t] = compute_duplicate_stats(t, embeddings, dates, top_k)
    intervals = [(0.0, 0.5), (0.5, 1.0)]
    iteration = 0
    while intervals and iteration < max_iter:
        new_intervals = []
        for t1, t2 in intervals:
            if t2 - t1 <= min_step:
                continue
            mid = (t1 + t2) / 2
            if mid not in results:
                results[mid] = compute_duplicate_stats(mid, embeddings, dates, top_k)
            r1 = results[t1]['removed']
            r2 = results[t2]['removed']
            r_mid = results[mid]['removed']
            delta = abs(r2 - r1) / max(1, abs(r1))
            if delta > change_threshold:
                new_intervals.append((t1, mid))
                new_intervals.append((mid, t2))
            delta_left = abs(r_mid - r1) / max(1, abs(r1))
            if delta_left > change_threshold:
                new_intervals.append((t1, mid))
            delta_right = abs(r2 - r_mid) / max(1, abs(r_mid))
            if delta_right > change_threshold:
                new_intervals.append((mid, t2))
        intervals = new_intervals
        iteration += 1
    df_results = pd.DataFrame([
        {'threshold': t, 'num_dup_clusters': res['num_dup_clusters'],
         'total_duplicates': res['total_duplicates'], 'removed': res['removed'],
         'avg_dup_size': res['avg_dup_size'], 'silhouette': res['silhouette']}
        for t, res in results.items()
    ]).sort_values('threshold').reset_index(drop=True)
    return df_results, results

def find_optimal_threshold(df_results, metric='removed', smooth_window=5):
    if len(df_results) > smooth_window:
        y = savgol_filter(df_results[metric].values, window_length=smooth_window, polyorder=2)
    else:
        y = df_results[metric].values
    x = df_results['threshold'].values
    dy = np.gradient(y, x)
    d2y = np.gradient(dy, x)
    idx_max = np.argmax(d2y)
    optimal_threshold = x[idx_max]
    return optimal_threshold, df_results.iloc[idx_max]

print("Запуск адаптивного поиска порога...")
df_adaptive, all_results = adaptive_threshold_search(
    embeddings, df.post_dttm,
    initial_step=0.1,
    min_step=0.001,
    change_threshold=0.05,
    max_iter=30,
    top_k=6
)

optimal_th, optimal_row = find_optimal_threshold(df_adaptive, metric='removed', smooth_window=5)
optimal_key = min(all_results.keys(), key=lambda k: abs(k - optimal_th))
print(f"Оптимальный порог (по точке перегиба): {optimal_th:.3f}")
print(f"При этом пороге: удалено {optimal_row['removed']:.0f} новостей, "
      f"кластеров дублей: {optimal_row['num_dup_clusters']:.0f}, "
      f"силуэт: {optimal_row['silhouette']:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax = axes[0, 0]
ax.plot(df_adaptive['threshold'], df_adaptive['removed'], 'o-', label='removed')
ax.axvline(optimal_th, color='red', linestyle='--', label=f'оптимальный {optimal_th:.3f}')
ax.set_xlabel('Порог'); ax.set_ylabel('Удалённых новостей'); ax.legend(); ax.grid(True)

ax = axes[0, 1]
ax.plot(df_adaptive['threshold'], df_adaptive['avg_dup_size'], 's-', color='green', label='avg cluster size')
ax.axvline(optimal_th, color='red', linestyle='--')
ax.set_xlabel('Порог'); ax.set_ylabel('Средний размер кластера дублей'); ax.legend(); ax.grid(True)

ax = axes[1, 0]
ax.plot(df_adaptive['threshold'], df_adaptive['num_dup_clusters'], '^-', color='orange', label='num clusters')
ax.axvline(optimal_th, color='red', linestyle='--')
ax.set_xlabel('Порог'); ax.set_ylabel('Количество кластеров дублей'); ax.legend(); ax.grid(True)

ax = axes[1, 1]
mask = df_adaptive['silhouette'] > -0.5
ax.plot(df_adaptive.loc[mask, 'threshold'], df_adaptive.loc[mask, 'silhouette'], 'd-', color='purple', label='silhouette')
ax.axvline(optimal_th, color='red', linestyle='--')
ax.set_xlabel('Порог'); ax.set_ylabel('Silhouette score'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig('adaptive_threshold_analysis.png', dpi=150)
plt.show()

df_adaptive.to_csv('adaptive_threshold_results.csv', index=False, encoding='utf-8-sig')
print("Результаты поиска сохранены в adaptive_threshold_results.csv")

experiments[f'modernbert_adaptive_{optimal_th:.3f}'] = all_results[optimal_key]['clusters']
print(f"Добавлен кластер для порога {optimal_th:.3f} в словарь experiments")

In [ ]:
# Ручная проверка: посмотрите кластеры выбранного метода до удаления.
SELECTED_METHOD = f'modernbert_adaptive_{optimal_th:.3f}'
df['duplicate_cluster'] = experiments[SELECTED_METHOD]
cluster_sizes = df.duplicate_cluster.value_counts()
sample_ids = cluster_sizes[cluster_sizes > 1].head(10).index
display(df[df.duplicate_cluster.isin(sample_ids)]
        .sort_values(['duplicate_cluster', 'post_dttm'])
        [['duplicate_cluster', 'source', 'post_dttm', 'data']])

In [ ]:
cluster_sizes = df['duplicate_cluster'].value_counts()
duplicate_clusters = cluster_sizes[cluster_sizes > 1]

print(f'Кластеров со схожими новостями: {len(duplicate_clusters)}')
print(f'Новостей внутри таких кластеров: {duplicate_clusters.sum()}')

cluster_id = duplicate_clusters.index[min(4, len(duplicate_clusters) - 1)]

print(f'Кластер {cluster_id}, новостей: {duplicate_clusters[cluster_id]}')

display(
    df.loc[df['duplicate_cluster'] == cluster_id,
           ['id', 'source', 'post_dttm', 'data']]
      .sort_values('post_dttm')
)

In [ ]:
ordered = df.sort_values(['duplicate_cluster', 'post_dttm']).copy()
posts_to_remove = ordered[ordered.duplicated('duplicate_cluster', keep='first')].copy()

print(f'К исключению: {len(posts_to_remove):,} постов')
display(posts_to_remove[['duplicate_cluster', 'id', 'source', 'post_dttm', 'data']]
        .sort_values(['duplicate_cluster', 'post_dttm'])
        .head(30))

posts_to_remove.to_csv(OUT_DIR / f'posts_to_remove_{SELECTED_METHOD}.csv',
                       index=False, encoding='utf-8-sig')

In [ ]:
try:
    import faiss
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])
    import faiss
import numpy as np

embeddings = embeddings.astype(np.float32)
faiss.normalize_L2(embeddings)

TOP_K = 2
SIMILARITY_THRESHOLD = 0.90

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

similarities, neighbor_ids = index.search(embeddings, TOP_K)

edges = []
for i in range(len(df)):
    for similarity, j in zip(similarities[i, 1:], neighbor_ids[i, 1:]):
        if i < j and similarity >= SIMILARITY_THRESHOLD:
            edges.append((i, int(j)))

faiss_clusters = clusters_from_edges(len(df), edges)
df_faiss = df.assign(duplicate_cluster=faiss_clusters)

news_deduplicated = (
    df_faiss.sort_values('post_dttm')
    .drop_duplicates('duplicate_cluster', keep='first')
)

posts_to_remove = (
    df_faiss.sort_values(['duplicate_cluster', 'post_dttm'])
    .loc[lambda x: x.duplicated('duplicate_cluster', keep='first')]
)

print(f'Найдено пар дублей: {len(edges):,}')
print(f'Было постов: {len(df_faiss):,}')
print(f'К удалению: {len(posts_to_remove):,}')
print(f'Осталось: {len(news_deduplicated):,}')

display(posts_to_remove[
    ['duplicate_cluster', 'source', 'post_dttm', 'data']
].head(30))

In [ ]:
deduplicated = (df.sort_values('post_dttm')
                  .drop_duplicates('duplicate_cluster', keep='first')
                  .drop(columns=['duplicate_cluster']))
duplicates_audit = (df[df.duplicate_cluster.duplicated(keep=False)]
                    .sort_values(['duplicate_cluster', 'post_dttm']))

deduplicated.to_csv(OUT_DIR / f'news_deduplicated_{SELECTED_METHOD}.csv', index=False, encoding='utf-8-sig')
duplicates_audit.to_csv(OUT_DIR / f'duplicate_clusters_{SELECTED_METHOD}.csv', index=False, encoding='utf-8-sig')
report.to_csv(OUT_DIR / 'hypotheses_comparison.csv', index=False, encoding='utf-8-sig')
print(f'Было: {len(df):,}; осталось: {len(deduplicated):,}; удалено: {len(df) - len(deduplicated):,}')

In [ ]:
import os
os.makedirs('/kaggle/working/output', exist_ok=True)
df_adaptive.to_csv('/kaggle/working/adaptive_threshold_results.csv', index=False)

In [ ]:
try:
    print(df_adaptive.head())
except NameError:
    print("❌ Переменная df_adaptive не определена.")

In [ ]:
#%% md
# Часть 2. Обогащение новостей метаданными для текстового энкодера
#
# Дальше строим то, что реально нужно в паре с моделью котировок (следующий ноутбук):
# на выходе — таблица (событие x тикер) с эмбеддингом, весом значимости, весом новизны
# и классом "времени действия" новости (+ ожидаемый half-life затухания эффекта).
#
# Что вошло и почему:
#   2.1 Тикеры — извлекаем упоминания и их позицию в тексте (нужно для 2.2-2.5).
#   2.2 Значимость (significance) — вес по размеру кластера дублей / числу уникальных
#       источников, ровно то, что вы предложили, с уточнением (см. ниже).
#   2.3 Новизна (novelty / Event Similarity Gap) — дополняет значимость: перепечатка в
#       первый час важнее, чем то же самое в 10-й раз через 3 недели.
#   2.4 Длительность эффекта (duration/decay) — новая часть по вашему запросу: смена
#       руководства — недели, разовое высказывание политика — часы.
#   2.5 Релевантность (REL) — нужна, чтобы понимать, ПО КАКОМУ тикеру считать вес, когда
#       в одной новости упомянуто несколько компаний.
#   2.6 Сборка финальной таблицы (событие x тикер) + экспорт эмбеддингов.
#
# Что НЕ вошло сюда (осознанно) и куда уехало:
#   - Регрессия новостей на волатильность/объём торгов IMOEX — задача СЛЕДУЮЩЕГО ноутбука
#     (модель котировок), а не текстового энкодера.
#   - Сравнение эмбеддинг-моделей, потоковая обработка, LLM-валидация кластеров — остаются
#     отдельными исследовательскими ноутбуками, здесь используется уже выбранная связка
#     RuModernBERT-small + адаптивный порог из Части 1.
#%%


In [ ]:
#%% md
## 2.1 Извлечение тикеров (с позицией упоминания — понадобится для REL в 2.5)
#%%
import requests

ENRICH_DIR = Path('news_encoder_results')
ENRICH_DIR.mkdir(exist_ok=True)

def load_moex_tickers():
    url = 'https://iss.moex.com/iss/engines/stock/markets/shares/securities.json'
    try:
        resp = requests.get(url, params={'iss.meta': 'off'}, timeout=20)
        resp.raise_for_status()
        payload = resp.json()['securities']
        idx = payload['columns'].index('SECID')
        tickers = sorted({row[idx] for row in payload['data'] if row[idx]})
        if tickers:
            return tickers
        raise ValueError('пустой список')
    except Exception as exc:
        print(f'MOEX ISS недоступен ({exc}), использую запасной список тикеров.')
        return ['SBER', 'GAZP', 'LKOH', 'GMKN', 'NVTK', 'ROSN', 'TATN', 'MGNT',
                'MTSS', 'YDEX', 'VTBR', 'CHMF', 'ALRS', 'PLZL', 'MOEX', 'AFLT',
                'PHOR', 'RUAL', 'POLY', 'SNGS']

MOEX_TICKERS = load_moex_tickers()
ticker_pattern = re.compile(r'(?:\$)?\b(' + '|'.join(re.escape(t) for t in MOEX_TICKERS) + r')\b')

def find_ticker_positions(text):
    text = str(text or '')
    return [(m.group(1), m.start()) for m in ticker_pattern.finditer(text)]

df['ticker_hits'] = df['data'].map(find_ticker_positions)
df['tickers_found'] = df['ticker_hits'].map(lambda hits: sorted({t for t, _ in hits}))

covered = df['tickers_found'].map(len).gt(0)
print(f'Новостей с тикером: {covered.sum():,} из {len(df):,} ({covered.mean():.1%})')

top_n = 20
ticker_counts = pd.Series([t for tickers in df['tickers_found'] for t in tickers]).value_counts()
if len(ticker_counts) > 0:
    fig, ax = plt.subplots(figsize=(9, max(4, min(top_n, len(ticker_counts)) * 0.3)))
    ticker_counts.head(top_n).sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Топ-{min(top_n, len(ticker_counts))} тикеров по числу упоминаний')
    plt.tight_layout()
    plt.savefig(ENRICH_DIR / 'ticker_distribution.png', dpi=150)
    plt.show()

In [ ]:
#%% md
## 2.2 Значимость новости (significance) — вес по кластеру дублей
# Как применяем: для каждой новости берём её кластер дублей (duplicate_cluster, уже посчитан
#     в Части 1 методом SELECTED_METHOD) и считаем:
#       significance_raw_count — сколько постов в кластере (буквально ваше предложение —
#           "размер кластера дублей как вес");
#       significance_source_count — сколько РАЗНЫХ каналов (source) перепечатали новость.
#     В качестве основного веса используем significance_source_count, а не raw_count:
#     один и тот же канал, повторивший пост 5 раз, не должен весить как 5 независимых
#     подтверждений важности — а разные каналы, покрывшие одно событие, весят. Обе метрики
#     сохраняются в таблицу, чтобы при желании переключиться на raw_count.
# Как оцениваем результат: смотрим распределение significance_score и топ-10 самых
#     "значимых" кластеров — глазами проверяем, что это действительно резонансные новости.
#%%
df['significance_raw_count'] = df.groupby('duplicate_cluster')['id'].transform('count')
df['significance_source_count'] = df.groupby('duplicate_cluster')['source'].transform('nunique')
df['significance_score'] = np.log1p(df['significance_source_count'])

fig, ax = plt.subplots(figsize=(9, 4))
df['significance_score'].hist(bins=30, ax=ax)
ax.set_title('Распределение significance_score (log1p от числа уникальных источников)')
ax.set_xlabel('significance_score')
plt.tight_layout()
plt.savefig(ENRICH_DIR / 'significance_hist.png', dpi=150)
plt.show()

top_clusters = (df[['duplicate_cluster', 'significance_source_count']]
                .drop_duplicates()
                .sort_values('significance_source_count', ascending=False)
                .head(10))
print('Топ-10 кластеров по числу уникальных источников:')
display(df[df.duplicate_cluster.isin(top_clusters.duplicate_cluster)]
        .sort_values(['duplicate_cluster', 'post_dttm'])
        [['duplicate_cluster', 'source', 'post_dttm', 'significance_source_count', 'data']]
        .groupby('duplicate_cluster').head(1))

In [ ]:
#%% md
## 2.3 Новизна новости (novelty / Event Similarity Gap)
# Что проверяем: помимо жёсткого порога дублей (SELECTED_METHOD), считаем МЯГКУЮ похожесть
#     (SOFT_SIM_THRESHOLD) — это ловит не только буквальные копии, но и "ту же историю
#     другими словами". Для новости считаем novelty_gap_days — сколько дней прошло с
#     ближайшей по времени мягко-похожей новости ПО ТОМУ ЖЕ ТИКЕРУ (если такой не было в
#     пределах WINDOW_DAYS — новость считается полностью новой, novelty_gap_days = WINDOW_DAYS).
# Как применяем (эффективно, без O(n^2) по каждому тикеру, в отличие от черновой версии
#     идеи в исследовательском ноутбуке): переиспользуем уже посчитанные ModernBERT-эмбеддинги
#     и ту же батчевую функцию top_k_edges, что и в Части 1, — считаем это ОДИН РАЗ для
#     всего корпуса с мягким порогом, затем фильтруем пары по пересечению множества тикеров
#     и раскладываем по времени (кто раньше — тот "источник" гэпа).
# Как оцениваем результат: гистограмма gap, доля "полностью новых" новостей, доля "stale"
#     (gap < STALE_GAP_DAYS) — используется дальше как is_stale-флаг в финальной таблице.
#%%
SOFT_SIM_THRESHOLD = 0.60
STALE_GAP_DAYS = 7

soft_edges = top_k_edges(embeddings, df.post_dttm, SOFT_SIM_THRESHOLD, n_neighbors=10)

ticker_sets = df['tickers_found'].map(set).to_numpy()
ts = df.post_dttm.to_numpy()

novelty_gap_days = np.full(len(df), float(WINDOW_DAYS))
for i, j in tqdm(soft_edges, desc='Разбор мягких пар для novelty gap'):
    if not (ticker_sets[i] & ticker_sets[j]):
        continue  # похожи по тексту, но про разные компании — не считаем повтором события
    if ts[i] <= ts[j]:
        earlier, later = i, j
    else:
        earlier, later = j, i
    gap = (ts[later] - ts[earlier]) / np.timedelta64(1, 'D')
    if gap < novelty_gap_days[later]:
        novelty_gap_days[later] = gap

df['novelty_gap_days'] = novelty_gap_days
df['is_stale'] = df['novelty_gap_days'] < STALE_GAP_DAYS

fig, ax = plt.subplots(figsize=(9, 4))
df['novelty_gap_days'].hist(bins=40, ax=ax)
ax.axvline(STALE_GAP_DAYS, color='red', linestyle='--', label=f'порог stale = {STALE_GAP_DAYS} дн.')
ax.set_title('Распределение novelty_gap_days (Event Similarity Gap)')
ax.set_xlabel('Дней с последней похожей новости по тому же тикеру')
ax.legend()
plt.tight_layout()
plt.savefig(ENRICH_DIR / 'novelty_gap_hist.png', dpi=150)
plt.show()

print(f'Доля "полностью новых" (gap = {WINDOW_DAYS}): {(df.novelty_gap_days >= WINDOW_DAYS).mean():.1%}')
print(f'Доля "устаревших" (gap < {STALE_GAP_DAYS} дн.): {df.is_stale.mean():.1%}')

In [ ]:
#%% md
## 2.4 Длительность эффекта новости (duration / decay) — новая часть по вашему запросу
# Что проверяем: разные типы новостей должны затухать в цене с разной скоростью — смена
#     гендиректора влияет на ожидания рынка неделями, а разовая цитата чиновника обычно
#     отыгрывается рынком за часы. Явно классифицируем каждую новость по категории события
#     и присваиваем ожидаемый half-life (время, за которое вес новости, по нашей модели,
#     падает вдвое).
# Как применяем: сначала быстрый детерминированный классификатор по ключевым словам
#     (RULE_CATEGORIES, приоритет сверху вниз — первое совпадение побеждает). Для новостей,
#     которые НЕ попали ни под одно правило — zero-shot классификатор в следующей ячейке.
# ВАЖНО: half-life ниже — экспертная эвристика, а не число, откалиброванное по фактической
#     реакции цены. Когда появится модель котировок — эти half-life стоит переоценить
#     эмпирически (например, через post-event decay самой |доходности|) и подставить сюда
#     откалиброванные значения.
#%%
# half_life_hours — экспертная эвристика:
#  - руководство/M&A/регуляторика: эффект тянется неделями (меняются ожидания по
#    стратегии/риску надолго) -> сотни часов;
#  - отчётность/дивиденды: рынок отыгрывает в первые дни, но есть post-earnings drift
#    (эффект известен по литературе о PEAD) -> порядка недели;
#  - макро/ЦБ: обычно отыгрывается за 1-2 торговых дня;
#  - разовые высказывания политиков/чиновников: короткоживущий шум, часы;
#  - рыночные комментарии/прогнозы аналитиков: почти мгновенно устаревают.
DURATION_HALF_LIFE_HOURS = {
    'руководство_и_структура': 720,   # смена CEO/собственника, реструктуризация, банкротство
    'сделки_ma':               336,   # слияния, поглощения, IPO, делистинг, продажа доли
    'регуляторика_санкции':    504,   # санкции, штрафы, суд, лицензии, запреты
    'отчетность_дивиденды':    120,   # финансовая отчётность, дивиденды, прибыль/выручка
    'макро_цб':                48,    # ставка ЦБ, инфляция, ВВП, курс рубля
    'заявления_политиков':     12,    # разовые высказывания/цитаты чиновников и политиков
    'рыночный_комментарий':    4,     # мнения аналитиков, прогнозы, технический анализ
    'прочее':                  24,    # запасной вариант по умолчанию
}

RULE_CATEGORIES = [
    ('руководство_и_структура', re.compile(r'\b(гендиректор|ген\.?\s?директор|совет директоров|'
        r'уволил\w*|уволен\w*|отставк\w*|назначен\w*\s+(на пост|главой|руководителем)|'
        r'реструктуризац\w*|банкрот\w*|сменил\w*\s+(руководство|президента))\b')),
    ('сделки_ma', re.compile(r'\b(слияни\w*|поглощени\w*|приобрет\w*|сделк\w*\s+по\s+продаже|'
        r'ipo|делистинг\w*|продал\w*\s+долю|купил\w*\s+долю|покупк\w*\s+актив\w*)\b')),
    ('регуляторика_санкции', re.compile(r'\b(санкци\w*|оштрафован\w*|штраф\w*|суд\w*\s+постановил|'
        r'лицензи\w*\s+(отозвана|отозван)|запрет\w*|цб\s+ввел|регулятор\w*\s+(запретил|ограничил))\b')),
    ('отчетность_дивиденды', re.compile(r'\b(отчетност\w*|отчётност\w*|финансов\w*\s+результат\w*|'
        r'выручк\w*|ebitda|чист\w*\s+прибыл\w*|дивиденд\w*|прогноз\w*\s+прибыл\w*)\b')),
    ('макро_цб', re.compile(r'\b(ключев\w*\s+ставк\w*|инфляц\w*|ввп|минфин|центробанк|цб\s+рф|'
        r'курс\s+рубля|курс\s+доллара)\b')),
    ('заявления_политиков', re.compile(r'\b(заявил\w*|сообщил\w*|по\s+словам|трамп|путин|байден|'
        r'президент\s+(сша|рф)?|министр\w*|пресс-секретар\w*)\b')),
    ('рыночный_комментарий', re.compile(r'\b(аналитик\w*\s+(считают|ожидают|прогнозируют)|'
        r'по\s+мнению\s+аналитик\w*|технический\s+анализ|прогноз\w*\s+курс\w*)\b')),
]

def rule_based_category(text):
    for label, pattern in RULE_CATEGORIES:
        if pattern.search(text):
            return label
    return None

df['duration_class_rule'] = df['text_normalized'].map(rule_based_category)
matched_share = df['duration_class_rule'].notna().mean()
print(f'Классифицировано по ключевым словам: {matched_share:.1%} новостей')

In [ ]:
#%% md
### Zero-shot дообработка новостей без совпадения по ключевым словам
# Как применяем: для новостей без совпадений по правилам прогоняем multilingual zero-shot
#     NLI-классификатор (mDeBERTa) по тем же категориям. Он медленнее, поэтому ограничиваем
#     объём ZERO_SHOT_MAX_ROWS — остальные без совпадений уходят в категорию 'прочее'.
# Как оцениваем результат: таблица покрытия категорий (доля новостей в каждой + её half-life)
#     и распределение классов на графике.
#%%
ZERO_SHOT_MAX_ROWS = 3000
unmatched_idx = df.index[df['duration_class_rule'].isna()]
print(f'Без совпадения по правилам: {len(unmatched_idx):,} новостей')

df['duration_class'] = df['duration_class_rule']

if len(unmatched_idx) > 0:
    try:
        from transformers import pipeline
        zero_shot = pipeline('zero-shot-classification', model='MoritzLaurer/mDeBERTa-v3-base-mnli-xnli',
                              device=0 if torch.cuda.is_available() else -1)
        candidate_labels = [c for c in DURATION_HALF_LIFE_HOURS if c != 'прочее']

        sample_idx = unmatched_idx
        if len(sample_idx) > ZERO_SHOT_MAX_ROWS:
            sample_idx = pd.Index(np.random.default_rng(42).choice(sample_idx, ZERO_SHOT_MAX_ROWS, replace=False))
            print(f'Слишком много новостей без правила — обрабатываю случайную выборку из {ZERO_SHOT_MAX_ROWS:,}, '
                  f'остальные получат категорию "прочее".')

        texts = df.loc[sample_idx, 'text_normalized'].str.slice(0, 512).tolist()
        preds = []
        for start in tqdm(range(0, len(texts), 16), desc='Zero-shot классификация'):
            batch = texts[start:start + 16]
            out = zero_shot(batch, candidate_labels=candidate_labels, multi_label=False)
            out = out if isinstance(out, list) else [out]
            preds.extend(o['labels'][0] for o in out)
        df.loc[sample_idx, 'duration_class'] = preds
    except Exception as exc:
        print(f'Zero-shot классификатор недоступен ({exc}) — оставшиеся новости получат категорию "прочее".')

df['duration_class'] = df['duration_class'].fillna('прочее')
df['half_life_hours'] = df['duration_class'].map(DURATION_HALF_LIFE_HOURS)

coverage = df['duration_class'].value_counts(normalize=True).rename('доля').to_frame()
coverage['half_life_hours'] = coverage.index.map(DURATION_HALF_LIFE_HOURS)
print(coverage)
coverage.to_csv(ENRICH_DIR / 'duration_class_coverage.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(9, 4))
df['duration_class'].value_counts().plot(kind='barh', ax=ax, color='darkorange')
ax.set_title('Распределение новостей по классу длительности эффекта')
ax.set_xlabel('Число новостей')
plt.tight_layout()
plt.savefig(ENRICH_DIR / 'duration_class_distribution.png', dpi=150)
plt.show()

In [ ]:
#%% md
### Функция затухания веса во времени (для передачи в модель котировок)
# decay_weight(t) = significance_score * 0.5^(t / half_life_hours) — экспоненциальное
# затухание с периодом полураспада half_life_hours. Модель котировок сможет на любой
# момент времени t получить актуальный вес новости, а не только на момент публикации.
#%%
def decay_weight(significance_score, half_life_hours, elapsed_hours):
    return significance_score * np.power(0.5, elapsed_hours / half_life_hours)

demo = (df.sort_values('significance_score', ascending=False)
          .drop_duplicates('duration_class')
          .head(4))
hours_range = np.linspace(0, 24 * 30, 200)

fig, ax = plt.subplots(figsize=(10, 5))
for _, row in demo.iterrows():
    weights = decay_weight(row.significance_score, row.half_life_hours, hours_range)
    ax.plot(hours_range / 24, weights, label=f'{row.duration_class} (half-life={row.half_life_hours}ч)')
ax.set_xlabel('Дней с публикации')
ax.set_ylabel('Вес новости')
ax.set_title('Пример затухания веса значимости по классам длительности')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ENRICH_DIR / 'decay_weight_demo.png', dpi=150)
plt.show()

In [ ]:
#%% md
## 2.5 Релевантность упоминания тикера (REL)
# Что проверяем: в одной новости может быть несколько тикеров, и не все упомянуты с
#     одинаковой значимостью — тикер в первом предложении важнее тикера, вскользь
#     упомянутого в конце длинного текста.
# Как применяем: REL = 100 * (0.6 * position_score + 0.4 * count_score), где position_score
#     тем выше, чем раньше первое упоминание, а count_score растёт (с насыщением) с числом
#     упоминаний тикера в тексте (используем уже сохранённые позиции из ticker_hits).
# Как оцениваем результат: распределение REL, доля пар (новость, тикер) с REL >= 70 —
#     именно эти пары дальше считаются "основными" при сборке финальной таблицы.
#%%
def compute_rel_scores(row):
    text_len = max(len(str(row['data'] or '')), 1)
    by_ticker = {}
    for ticker, pos in row['ticker_hits']:
        by_ticker.setdefault(ticker, []).append(pos)
    scores = {}
    for ticker, positions in by_ticker.items():
        position_score = max(0.0, 1 - min(positions) / text_len)
        count_score = 1 - 1 / (1 + len(positions))
        scores[ticker] = round(100 * (0.6 * position_score + 0.4 * count_score), 1)
    return scores

df['ticker_rel_scores'] = df.apply(compute_rel_scores, axis=1)

all_rel = [rel for scores in df['ticker_rel_scores'] for rel in scores.values()]
if all_rel:
    fig, ax = plt.subplots(figsize=(9, 4))
    pd.Series(all_rel).hist(bins=30, ax=ax)
    ax.set_title('Распределение REL-скора по парам (новость, тикер)')
    ax.set_xlabel('REL (0-100)')
    plt.tight_layout()
    plt.savefig(ENRICH_DIR / 'rel_score_hist.png', dpi=150)
    plt.show()

REL_THRESHOLD = 70
share_high_rel = np.mean([r >= REL_THRESHOLD for r in all_rel]) if all_rel else 0
print(f'Пар с REL >= {REL_THRESHOLD}: {share_high_rel:.1%}')

In [ ]:
#%% md
## 2.6 Сборка финальной таблицы признаков (событие x тикер)
# Как применяем: для каждого кластера дублей (события) берём представителя — САМЫЙ РАННИЙ
#     пост (момент, когда история впервые появилась в потоке). Тикеры и REL берём
#     ОБЪЕДИНЕНИЕМ по всем постам кластера (разные перепечатки могли упоминать тикер
#     по-разному), для REL внутри кластера берём МАКСИМУМ по всем постам этого тикера.
# Как оцениваем результат: итоговая таблица — одна строка на пару (событие, тикер), с
#     эмбеддингом представителя (через embedding_row_id), готова к склейке с ценами.
#%%
representative_idx = (df.sort_values(['duplicate_cluster', 'post_dttm'])
                        .drop_duplicates('duplicate_cluster', keep='first').index)
representatives = df.loc[representative_idx].set_index('duplicate_cluster')

cluster_rel = {}
for cluster_id, group in df.groupby('duplicate_cluster'):
    merged = {}
    for scores in group['ticker_rel_scores']:
        for ticker, rel in scores.items():
            merged[ticker] = max(merged.get(ticker, 0), rel)
    cluster_rel[cluster_id] = merged

rows = []
for cluster_id, rel_map in cluster_rel.items():
    if not rel_map:
        continue
    rep = representatives.loc[cluster_id]
    for ticker, rel in rel_map.items():
        rows.append({
            'event_id': cluster_id,
            'ticker': ticker,
            'event_dttm': rep['post_dttm'],
            'source': rep['source'],
            'rel_score': rel,
            'significance_score': rep['significance_score'],
            'significance_raw_count': rep['significance_raw_count'],
            'significance_source_count': rep['significance_source_count'],
            'novelty_gap_days': rep['novelty_gap_days'],
            'is_stale': rep['is_stale'],
            'duration_class': rep['duration_class'],
            'half_life_hours': rep['half_life_hours'],
            'embedding_row_id': rep['_row_id'],
            'text_preview': str(rep['data'])[:200],
        })

news_event_features = pd.DataFrame(rows).sort_values('event_dttm').reset_index(drop=True)
print(f'Итоговая таблица: {len(news_event_features):,} строк '
      f'({news_event_features.event_id.nunique():,} событий, {news_event_features.ticker.nunique():,} тикеров)')
display(news_event_features.head(10))

In [ ]:
#%% md
## 2.7 Экспорт финальных артефактов
# Сохраняем:
#   - news_event_features.csv — таблица (событие x тикер) для склейки с ценами;
#   - event_embeddings.npy — эмбеддинги ModernBERT, выровненные по embedding_row_id;
#   - краткий словарь данных (какая колонка что значит) — для быстрого онбординга
#     следующего ноутбука (модель котировок).
#%%
FINAL_DIR = Path('text_encoder_output')
FINAL_DIR.mkdir(exist_ok=True)

news_event_features.to_csv(FINAL_DIR / 'news_event_features.csv', index=False, encoding='utf-8-sig')
np.save(FINAL_DIR / 'event_embeddings.npy', embeddings)

data_dictionary = pd.DataFrame([
    ('event_id', 'ID кластера дублей (номер "истории")'),
    ('ticker', 'Тикер MOEX, к которому относится строка'),
    ('event_dttm', 'Время публикации самого раннего поста в кластере'),
    ('source', 'Канал-источник самого раннего поста'),
    ('rel_score', '0-100: насколько заметно упомянут тикер в тексте (позиция + число упоминаний)'),
    ('significance_score', 'log1p(число уникальных источников в кластере) — основной вес значимости'),
    ('significance_raw_count', 'Сырое число постов в кластере (включая повторы одного канала)'),
    ('significance_source_count', 'Число уникальных каналов, перепечатавших новость'),
    ('novelty_gap_days', 'Дней с последней похожей новости по этому тикеру (Event Similarity Gap)'),
    ('is_stale', 'True, если novelty_gap_days < 7 — новость, скорее всего, уже отыграна рынком'),
    ('duration_class', 'Категория события (см. DURATION_HALF_LIFE_HOURS)'),
    ('half_life_hours', 'Экспертная оценка периода полураспада эффекта новости, часов'),
    ('embedding_row_id', 'Индекс строки в event_embeddings.npy для эмбеддинга представителя события'),
    ('text_preview', 'Первые 200 символов текста представителя (для отладки/аудита)'),
], columns=['column', 'description'])
data_dictionary.to_csv(FINAL_DIR / 'data_dictionary.csv', index=False, encoding='utf-8-sig')

print(f'Событий: {news_event_features.event_id.nunique():,}')
print(f'Строк (событие x тикер): {len(news_event_features):,}')
print(f'Эмбеддингов сохранено: {embeddings.shape[0]:,} x {embeddings.shape[1]}')
display(data_dictionary)

In [ ]:
print('Текстовый энкодер готов. Артефакты в', FINAL_DIR.resolve())
print('Следующий шаг: модель котировок — принимает news_event_features.csv + event_embeddings.npy,')
print('строит временной ряд по каждому тикеру и учится предсказывать движение цены,')
print('используя significance_score * decay_weight(half_life_hours) как вес каждой новости во времени.')